# Agent-to-Agent (A2A)

UC3: Ein Rechercheur-Agent fragt einen BigQuery-Analysten über das A2A-Protokoll, beide als eigene Engines auf Agent Runtime und beide mit eigener Agent Identity. Der Analyst hat das BigQuery-Recht, der Rechercheur nur das Recht, den Analysten aufzurufen.

In [ ]:
# @title Installs
############################################

# Google Cloud Vertex AI (Agent Engine), Google ADK und das A2A-SDK installieren
# Feste Versionen statt --upgrade: Der Deploy pinnt dieselben Versionen, damit das Pickle in der Cloud passt
%pip install --quiet "google-cloud-aiplatform[agent_engines,adk]==2.6.0" "google-adk==2.13.0" "google-genai==2.33.0" "a2a-sdk[http-server]==1.1.2"

# Versionen kurz prüfen
%pip show google-adk google-cloud-aiplatform a2a-sdk

In [ ]:
# @title Imports
############################################

import logging
import os
import subprocess
import uuid
import warnings
from importlib.metadata import version

import google.auth
import google.auth.transport.requests
import httpx
import vertexai
from a2a.server.context import ServerCallContext
from a2a.types import AgentCard, AgentSkill, Message, Part, Role, SendMessageRequest, TaskState
from google.adk.a2a.executor.a2a_agent_executor import A2aAgentExecutor
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.cloud import bigquery
from google.protobuf import json_format
from vertexai import types
from vertexai.agent_engines import AdkApp
from vertexai.agent_engines.templates.a2a import A2aAgent, create_agent_card

warnings.filterwarnings('ignore')

print("✅ Alle Imports erfolgreich geladen!")

In [ ]:
# @title Environmental Variables & GCP Setup
############################################

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"  # Agent Engine Region
STAGING_BUCKET = "gs://deltorobarba-agent-staging"
GCS_DIR = "agent_a2a"  # eigener Ordner im Bucket: die Pickles der anderen Notebooks bleiben unberührt
MODEL = "gemini-3.8-flash"

# Nach dem ersten Deploy hier eintragen, damit ein zweiter Lauf keine weitere Engine anlegt
ANALYST_ENGINE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/7294320942241546240"  # A2A-Server: der BigQuery-Analyst (Teil A)
RESEARCHER_ENGINE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/68295340125585408"  # A2A-Client: der Rechercheur (Teil B)

# These tell the underlying google-genai client to use Vertex AI (Gemini 3.8 Flash uses 'global')
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# gcloud ohne Fortschritts-Spinner: Jupyter kann dessen Neuzeichnen nicht darstellen
os.environ["CLOUDSDK_CORE_INTERACTIVE_UX_STYLE"] = "OFF"

# Initialize Vertex AI SDK & Client (v1beta1: nötig für identity_type und den A2A-Endpunkt)
# vertexai.init ist hier Pflicht: A2aAgent liest Projekt und Region beim Erzeugen daraus
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION, http_options=dict(api_version="v1beta1"))

# SDK-Fortschritt bei Deploys anzeigen: das SDK loggt LRO und Resource-Namen auf INFO, was Jupyter sonst verschluckt
_log = logging.getLogger("vertexai_genai.agentengines")
_log.setLevel(logging.INFO)
_log.propagate = False
if not _log.handlers:
    _log.addHandler(logging.StreamHandler())

print("✅ Erfolgreich über Application Default Credentials (ADC) verbunden.")

##### <font color="blue">*Agent-to-Agent (A2A)*

Bei MCP benutzt ein Agent ein **Werkzeug**: Er ruft eine Funktion auf und bekommt Daten zurück. Bei A2A fragt ein Agent einen **Kollegen**: einen anderen Agent mit eigenem Modell, eigenen Tools und (hier) eigener Identität. Der Kollege entscheidet selbst, wie er die Aufgabe löst.

```mermaid
%%{init: {'themeVariables': {'fontSize': '13px'}, 'flowchart': {'nodeSpacing': 20, 'rankSpacing': 28, 'padding': 6}}}%%
flowchart LR
    U(["Du"]) -->|"query"| R["researcher<br/>Agent Identity R"]
    R -->|"A2A: Agent Card lesen,<br/>Message senden"| A["bigquery_analyst<br/>Agent Identity A"]
    A -->|"Tool, eigene Identität"| B[("BigQuery")]
```

**Begriffe des Protokolls** (A2A 1.0, Paket `a2a-sdk`)

| Begriff | Was es ist | Hier |
|---|---|---|
| **Agent Card** | Steckbrief des Agents als JSON: Name, Beschreibung, Fähigkeiten (*Skills*), Endpunkt, Protokoll | Der Analyst beschreibt sich als „BigQuery Analyst“ mit dem Skill „US baby names statistics“ |
| **Message** | eine Nachricht mit *Parts* (Text, Daten, Dateien) und einer Rolle (`ROLE_USER` oder `ROLE_AGENT`) | die Frage des Rechercheurs |
| **Task** | der Auftrag, den der Server aus der Message macht, mit Status (`TASK_STATE_WORKING`, `…_COMPLETED`, `…_FAILED`) | ein Task pro Frage |
| **Artifact** | das Ergebnis eines Tasks | die Antwort des Analysten mit den Zahlen |

**A2A auf Agent Runtime**

* Den Server-Teil liefert die Vorlage `A2aAgent` des Vertex SDK: Sie verpackt einen ADK-Agent (über `A2aAgentExecutor`) als A2A-Server.
* Der Endpunkt liegt auf einer **Google-API**: `https://us-central1-aiplatform.googleapis.com/v1beta1/<Engine>/a2a`, die Agent Card unter `…/a2a/v1/card`. Anmeldung mit einem normalen Google-**Access-Token**, Protokoll HTTP+JSON.
* Deshalb kann eine Agent Identity den Analysten aufrufen. Bei einem privaten Cloud-Run-Dienst ging das nicht (`identity.ipynb`, UC4: Cloud Run lehnt das ID-Token der Agent Identity mit 401 ab).

**Ablauf:** Teil A baut den Analysten als A2A-Server (lokal testen, deployen, BigQuery-Recht nur für ihn). Teil B baut den Rechercheur als A2A-Client (lokal testen, deployen) und klärt, welches Recht er braucht, um den Analysten aufzurufen.

##### <font color="blue">*Teil A: Der Analyst als A2A-Server*

In [ ]:
# @title A1. Analyst definieren (Agent mit BigQuery-Tool)
############################################
# Derselbe Analyst wie in identity.ipynb (UC4): Das Tool läuft als die Identität, unter der der Agent läuft.

ANALYST_INSTRUCTION = """
You are a data analyst with access to BigQuery through the tool 'query_public_dataset'.

SQL RULES:
- ALWAYS use SUM(number) and GROUP BY name to get totals.
- ALWAYS wrap table names in backticks: `bigquery-public-data.usa_names.usa_1910_2013`.
- If the tool returns an error, report the error verbatim and do NOT answer with numbers from memory.
"""


def query_public_dataset(sql_query: str) -> dict:
    """Run a read-only SQL query against BigQuery public datasets and return up to 20 rows.

    Args:
        sql_query: A BigQuery Standard SQL SELECT statement.
    """
    from google.cloud import bigquery  # erst hier importieren: im Pickle steckt nur die Funktion

    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE"]
    if any(cmd in sql_query.upper() for cmd in forbidden):
        return {"error": "Modification queries are not allowed. Read-only access."}
    try:
        job = bigquery.Client(project=PROJECT_ID).query(sql_query)
        return {"rows": [dict(row) for row in job.result(max_results=20)]}
    except Exception as e:
        return {"error": f"BigQuery: {e}"}


def build_analyst() -> LlmAgent:
    return LlmAgent(name="bigquery_analyst", model=MODEL, instruction=ANALYST_INSTRUCTION, tools=[query_public_dataset])


print("✅ build_analyst() definiert, Tool: query_public_dataset")

In [ ]:
# @title A2. Agent Card und A2A-Server (A2aAgent)
############################################
# Die Agent Card ist der Steckbrief, den andere Agents lesen. A2aAgent baut daraus den A2A-Server; die Arbeit
# macht der ADK-Agent, verbunden über A2aAgentExecutor (übersetzt A2A-Messages in ADK-Läufe und zurück).


def analyst_executor_builder():
    """Läuft in set_up(), also erst in der Runtime (bzw. beim lokalen Test), nicht beim Pickeln."""
    # set_up() setzt GOOGLE_CLOUD_LOCATION auf die Region der Engine (us-central1). Gemini 3.8 Flash gibt es nur
    # auf 'global'; ohne diese Zeile antwortet Gemini mit 404 "Publisher model … was not found".
    os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
    runner = Runner(app_name="bigquery_analyst", agent=build_analyst(), session_service=InMemorySessionService())
    return A2aAgentExecutor(runner=runner)


ANALYST_CARD = create_agent_card(
    agent_name="BigQuery Analyst",
    description="Answers questions about US baby names with SQL on BigQuery public data.",
    skills=[AgentSkill(
        id="baby_names",
        name="US baby names statistics",
        description="Top names per year and gender from bigquery-public-data.usa_names.usa_1910_2013",
        tags=["bigquery", "sql", "statistics"],
        examples=["What were the top 3 female names in the USA in 1980?"],
    )],
)


def build_analyst_a2a() -> A2aAgent:
    """Jedes Mal frisch bauen: Das lokal gestartete Objekt ist ein anderes als das deployte."""
    return A2aAgent(agent_card=ANALYST_CARD, agent_executor_builder=analyst_executor_builder)


print(json_format.MessageToJson(ANALYST_CARD))

In [ ]:
# @title A3. Lokaler Test des A2A-Servers (als du, ohne HTTP)
############################################
# set_up() baut den Server so, wie es die Runtime tut; on_message_send ist die A2A-Operation "Nachricht senden".
# Die Antwort ist ein Task: Status, Ergebnis (Artifact) und Verlauf (History, darin der Tool-Aufruf).

A2A_QUESTION = "What were the top 3 female names in the USA in 1980?"


def new_message(text: str) -> SendMessageRequest:
    return SendMessageRequest(message=Message(role=Role.ROLE_USER, message_id=str(uuid.uuid4()), parts=[Part(text=text)]))


def show_task(task):
    """Status, Tool-Aufrufe aus der History und Text der Artifacts eines A2A-Tasks."""
    print("Status:", TaskState.Name(task.status.state))
    for msg in task.history:
        for part in msg.parts:
            data = json_format.MessageToDict(part.data) if part.HasField("data") else {}
            if data.get("name") and data.get("args"):
                print(f"🔧 Tool-Aufruf im Analysten: {data['name']}")
    for artifact in task.artifacts:
        for part in artifact.parts:
            if part.text:
                print(f"📄 Artifact: {part.text}")


local_analyst = build_analyst_a2a()
local_analyst.set_up()
show_task(await local_analyst.on_message_send(new_message(A2A_QUESTION), ServerCallContext()))

In [ ]:
# @title A4. Analyst auf Agent Runtime deployen, mit Agent Identity (⚠️ --> once only!)
############################################
# Wie in identity.ipynb: identity_type geht nur beim create. Danach ANALYST_ENGINE_NAME oben eintragen.

ANALYST_CONFIG = {
    "display_name": "A2A BQ Analyst",
    "description": "BigQuery analyst as A2A server, with Agent Identity",
    "staging_bucket": STAGING_BUCKET,
    "gcs_dir_name": f"{GCS_DIR}/analyst",
    # Gleiche Versionen wie lokal: der Agent wird hier gepickelt und in der Cloud entpickelt
    "requirements": [
        f"google-cloud-aiplatform[agent_engines,adk]=={version('google-cloud-aiplatform')}",
        f"google-adk=={version('google-adk')}",
        f"google-genai=={version('google-genai')}",
        f"a2a-sdk[http-server]=={version('a2a-sdk')}",  # Server-Teil von A2A (Starlette-Routen)
        f"google-cloud-bigquery=={version('google-cloud-bigquery')}",
        f"cloudpickle=={version('cloudpickle')}",
        f"pydantic=={version('pydantic')}",
    ],
    "env_vars": {"GOOGLE_CLOUD_LOCATION": "global"},
    "identity_type": types.IdentityType.AGENT_IDENTITY,
}

if ANALYST_ENGINE_NAME:
    print("Engine existiert schon:", ANALYST_ENGINE_NAME)
else:
    analyst_engine = client.agent_engines.create(agent=build_analyst_a2a(), config=ANALYST_CONFIG)
    ANALYST_ENGINE_NAME = analyst_engine.api_resource.name
    print(f"\n✅ Engine angelegt: {ANALYST_ENGINE_NAME}")
    print("   👉 oben in 'Environmental Variables & GCP Setup' als ANALYST_ENGINE_NAME eintragen")

ANALYST_PRINCIPAL = f"principal://{client.agent_engines.get(name=ANALYST_ENGINE_NAME).api_resource.spec.effective_identity}"
# Der A2A-Endpunkt des Analysten: eine Google-API, keine eigene Domain
ANALYST_A2A_URL = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{ANALYST_ENGINE_NAME}/a2a"
print("Identität:   ", ANALYST_PRINCIPAL)
print("A2A-Endpunkt:", ANALYST_A2A_URL)

In [ ]:
# @title A5. BigQuery-Recht NUR für die Identität des Analysten (⚠️ --> once only!)
############################################
# Wie UC4 in identity.ipynb: ohne dieses Recht meldet das Tool 403 bigquery.jobs.create.
# Der Rechercheur bekommt dieses Recht NICHT: Er soll BigQuery nur über den Analysten erreichen.

!gcloud projects add-iam-policy-binding {PROJECT_ID} --member="{ANALYST_PRINCIPAL}" --role="roles/bigquery.jobUser" --condition=None --quiet > /dev/null && echo "✅ roles/bigquery.jobUser für den Analysten"
print("IAM-Propagation dauert ein bis zwei Minuten.")

In [ ]:
# @title A6. Test: du fragst den deployten Analysten per A2A
############################################
# Zwei Schritte des Protokolls: 1. Agent Card lesen (GET …/a2a/v1/card), 2. Message senden (SDK: on_message_send).
# Angemeldet wird mit deinem Access-Token; der Analyst ruft BigQuery dann als SEINE Identität auf.


def google_headers() -> dict:
    """Authorization-Header mit einem frischen Access-Token aus ADC (lokal: du, deployed: die Agent Identity)."""
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(google.auth.transport.requests.Request())
    return {"Authorization": f"Bearer {creds.token}"}


card = httpx.get(f"{ANALYST_A2A_URL}/v1/card", headers=google_headers(), timeout=60).json()
print("Agent Card:", card["name"], "|", card["description"])
print("  Skills:  ", [s["name"] for s in card.get("skills", [])])
print("  Endpunkt:", [i["url"] for i in card.get("supportedInterfaces", [])])

remote_analyst = client.agent_engines.get(name=ANALYST_ENGINE_NAME)
for chunk in await remote_analyst.on_message_send(request=new_message(A2A_QUESTION)):
    if chunk.HasField("task"):
        show_task(chunk.task)

##### <font color="blue">*Teil B: Der Rechercheur als A2A-Client*

Der Rechercheur beantwortet Fragen nicht selbst, wenn es um Zahlen geht: Er fragt den Analysten per A2A. Das Tool `ask_bigquery_analyst` zeigt die Schritte des Protokolls offen im Code: Token holen, Agent Card lesen, Client aus der Card bauen, Message senden, Artifact lesen. (ADK hat dafür auch `RemoteA2aAgent`; das übergibt das Gespräch aber ganz an den anderen Agent und lässt sich mit der Anmeldung an Google schwerer pickeln.)

Wer meldet sich wo an:

| Schritt | Wer | Womit | Was geprüft wird |
|---|---|---|---|
| Du → Rechercheur | du | dein Access-Token | dein Recht auf die Engine des Rechercheurs |
| Rechercheur → Analyst (A2A) | lokal du, deployed die Agent Identity des Rechercheurs | Access-Token aus ADC | das Recht dieser Identität auf die Engine des Analysten |
| Analyst → BigQuery | die Agent Identity des Analysten | Access-Token der Runtime | `roles/bigquery.jobUser` (A5) |

In [ ]:
# @title B1. Rechercheur definieren (Agent mit A2A-Client als Tool)
############################################
# ANALYST_A2A_URL kommt aus A4. Die Funktion wird mit dem Agent gepickelt; die URL steckt dann als Wert darin.

RESEARCHER_INSTRUCTION = """
You are a research assistant. For any question about US baby name statistics, ask your colleague with the tool
'ask_bigquery_analyst' instead of answering from memory. Pass the question in plain English.
Then answer the user in at most three sentences and say that the numbers come from the BigQuery Analyst.
If the tool returns an error, report the error verbatim and do NOT answer with numbers from memory.
"""


async def ask_bigquery_analyst(question: str) -> str:
    """Ask the BigQuery Analyst agent (via the A2A protocol) a question about US baby name statistics.

    Args:
        question: The question in plain English.
    """
    # Imports hier: die Funktion läuft deployed in der Runtime des Rechercheurs
    import uuid

    import google.auth
    import google.auth.transport.requests
    import httpx
    from a2a.client import ClientConfig, ClientFactory
    from a2a.types import AgentCard, Message, Part, Role, SendMessageRequest
    from a2a.utils.constants import TransportProtocol
    from google.auth.transport import mtls
    from google.protobuf import json_format

    try:
        # 1. Anmelden: Access-Token aus ADC (lokal du, deployed die Agent Identity des Rechercheurs)
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
        creds.refresh(google.auth.transport.requests.Request())
        # Das Token einer Agent Identity gilt nur zusammen mit dem Zertifikat ihrer Runtime (Context-Aware Access).
        # get_default_ssl_context() lädt dieses Zertifikat (lokal: None); dann muss der Aufruf an den
        # mTLS-Endpunkt gehen. Ohne beides antwortet Google mit 401, bevor überhaupt Rechte geprüft werden.
        ssl_context = mtls.get_default_ssl_context()
        a2a_url = ANALYST_A2A_URL
        if ssl_context is not None:
            a2a_url = a2a_url.replace("-aiplatform.googleapis.com", "-aiplatform.mtls.googleapis.com")
        async with httpx.AsyncClient(headers={"Authorization": f"Bearer {creds.token}"},
                                     verify=ssl_context or True, timeout=300) as http:
            # 2. Agent Card lesen: Name, Skills und Endpunkt des Analysten
            response = await http.get(f"{a2a_url}/v1/card")
            response.raise_for_status()
            card = json_format.ParseDict(response.json(), AgentCard(), ignore_unknown_fields=True)
            # Die Card nennt den normalen Endpunkt; wir bleiben auf dem, über den wir angemeldet sind
            card.supported_interfaces[0].url = a2a_url
            # 3. A2A-Client aus der Card bauen (HTTP+JSON, wie die Card es anbietet) und die Frage als Message senden
            config = ClientConfig(httpx_client=http, supported_protocol_bindings=[TransportProtocol.HTTP_JSON])
            a2a_client = ClientFactory(config).create(card)
            message = Message(role=Role.ROLE_USER, message_id=str(uuid.uuid4()), parts=[Part(text=question)])
            answers = []
            async for chunk in a2a_client.send_message(SendMessageRequest(message=message)):
                # 4. Antwort lesen: Text aus den Artifacts des Tasks (oder aus einer direkten Message)
                if chunk.HasField("task"):
                    answers += [part.text for artifact in chunk.task.artifacts for part in artifact.parts if part.text]
                elif chunk.HasField("message"):
                    answers += [part.text for part in chunk.message.parts if part.text]
            return "\n".join(answers) or "The analyst returned no text."
    except httpx.HTTPStatusError as e:
        # Status und Begründung von Google zurückgeben: 401 = Anmeldung gescheitert, 403 = Recht fehlt.
        # Nur den Text der Meldung, nicht das JSON drumherum: Mit geschweiften Klammern im Tool-Ergebnis brach
        # der Stream-Parser des SDK beim Test des deployten Agents ab ("Failed to parse response as JSON").
        try:
            detail = e.response.json()["error"]["message"].split(" Remediate")[0]
        except Exception:
            detail = e.response.reason_phrase
        return f"A2A error: HTTP {e.response.status_code}: {detail}"
    except Exception as e:
        # Fehler als Text zurückgeben: dann greift die Regel "keine Zahlen aus dem Gedächtnis"
        return f"A2A error: {type(e).__name__}: {e}"


def build_researcher() -> LlmAgent:
    """Jedes Mal frisch bauen: Das lokal getestete Objekt ist ein anderes als das deployte."""
    return LlmAgent(name="researcher", model=MODEL, instruction=RESEARCHER_INSTRUCTION, tools=[ask_bigquery_analyst])


print("✅ build_researcher() definiert, A2A-Ziel:", ANALYST_A2A_URL)

In [ ]:
# @title B2. Lokaler Test: du → Rechercheur (lokal) → A2A → Analyst (deployed)
############################################
# Der Rechercheur läuft hier als du; der Analyst läuft in der Cloud als seine eigene Identität.

RESEARCH_QUESTION = "How popular was the name Jennifer in 1980 compared to Amanda and Jessica?"


async def ask_with_tools(app, message: str, user_id: str = "deltorobarba"):
    """Zeigt Tool-Aufruf, Tool-Antwort (gekürzt) und Antwort des Agents; gleich für lokal und deployed."""
    session = await app.async_create_session(user_id=user_id)
    async for event in app.async_stream_query(user_id=user_id, session_id=session["id"], message=message):
        if event.get("error_message"):
            print(f"❌ [{event.get('author')}] {event.get('error_code')}: {event['error_message']}")
        for part in event.get("content", {}).get("parts", []):
            if "function_call" in part:
                print(f"🔧 Tool-Aufruf: {part['function_call']['name']}({part['function_call'].get('args')})")
            elif "function_response" in part:
                print(f"📦 Antwort des Analysten: {str(part['function_response'].get('response'))[:500]}")
            elif "text" in part and not part.get("thought"):
                print(f"🕵️ {part['text']}")


await ask_with_tools(AdkApp(agent=build_researcher()), RESEARCH_QUESTION)

In [ ]:
# @title B3. Rechercheur auf Agent Runtime deployen, mit Agent Identity (⚠️ --> once only!)
############################################
# Ein normaler ADK-Agent (AdkApp), kein A2A-Server: Er wird von dir aufgerufen und ist selbst nur A2A-Client.

RESEARCHER_CONFIG = {
    "display_name": "A2A Researcher",
    "description": "Research agent that asks the BigQuery Analyst via A2A, with Agent Identity",
    "staging_bucket": STAGING_BUCKET,
    "gcs_dir_name": f"{GCS_DIR}/researcher",
    "requirements": [
        f"google-cloud-aiplatform[agent_engines,adk]=={version('google-cloud-aiplatform')}",
        f"google-adk=={version('google-adk')}",
        f"google-genai=={version('google-genai')}",
        f"a2a-sdk=={version('a2a-sdk')}",  # nur der Client-Teil
        f"httpx=={version('httpx')}",
        f"google-auth=={version('google-auth')}",  # get_default_ssl_context() für mTLS in der Runtime
        f"cloudpickle=={version('cloudpickle')}",
        f"pydantic=={version('pydantic')}",
    ],
    "env_vars": {"GOOGLE_CLOUD_LOCATION": "global"},
    "identity_type": types.IdentityType.AGENT_IDENTITY,
}

if RESEARCHER_ENGINE_NAME:
    print("Engine existiert schon:", RESEARCHER_ENGINE_NAME)
else:
    researcher_engine = client.agent_engines.create(agent=AdkApp(agent=build_researcher()), config=RESEARCHER_CONFIG)
    RESEARCHER_ENGINE_NAME = researcher_engine.api_resource.name
    print(f"\n✅ Engine angelegt: {RESEARCHER_ENGINE_NAME}")
    print("   👉 oben in 'Environmental Variables & GCP Setup' als RESEARCHER_ENGINE_NAME eintragen")

RESEARCHER_PRINCIPAL = f"principal://{client.agent_engines.get(name=RESEARCHER_ENGINE_NAME).api_resource.spec.effective_identity}"
print("Identität:", RESEARCHER_PRINCIPAL)

In [ ]:
# @title B4. Test: du → Rechercheur (deployed) → A2A → Analyst (deployed)
############################################
# Jetzt ruft die Agent Identity des Rechercheurs den Analysten auf, nicht mehr du.

await ask_with_tools(client.agent_engines.get(name=RESEARCHER_ENGINE_NAME), RESEARCH_QUESTION)

In [ ]:
# @title B5. Aufrufrecht für den Rechercheur (⚠️ --> erst entscheiden, dann ausführen)
############################################
# Getestet am 2026-10-26: Ein Recht NUR auf die Engine des Analysten lässt sich nicht vergeben.
#  - Engines haben keine eigene IAM-Policy (getIamPolicy auf die Engine: 404).
#  - IAM-Bedingungen auf resource.name wirken bei Vertex AI nicht: mit `resource.name == "<Engine>"` und mit
#    `resource.name.endsWith("/reasoningEngines/<ID>")` blieb es nach mehreren Minuten bei 403. Die IAM-Doku listet
#    Vertex AI nicht unter den Diensten, die Ressourcen-Attribute in Bedingungen kennen.
# Was bleibt: eine eigene Rolle mit nur den zwei Rechten, die A2A braucht (reasoningEngines.get und .query), auf
# Projektebene. Folge: Der Rechercheur darf damit JEDE Engine im Projekt aufrufen, auch Pipeline, MCP-Analyst und
# "Agent Identity". Das ist weniger als aiplatform.viewer oder aiplatform.expressUser (die enthalten beide Rechte
# und viele weitere), aber nicht auf eine Engine beschränkt. agentDefaultAccess enthält keines der beiden Rechte.
# subprocess statt !gcloud: so bleiben die Argumente ohne Shell-Quoting lesbar.

A2A_GRANT = True  # am 2026-10-26 vom Nutzer entschieden: Rechercheur darf Engines im Projekt aufrufen

A2A_ROLE = f"projects/{PROJECT_ID}/roles/a2aCaller"
if not A2A_GRANT:
    print("Kein Recht vergeben (A2A_GRANT = False). B6 endet deshalb mit 403.")
else:
    role_exists = subprocess.run(["gcloud", "iam", "roles", "describe", "a2aCaller", f"--project={PROJECT_ID}"],
                                 capture_output=True).returncode == 0
    if not role_exists:
        subprocess.run(["gcloud", "iam", "roles", "create", "a2aCaller", f"--project={PROJECT_ID}", "--title=A2A Caller",
                        "--description=Read the agent card of and send messages to Agent Runtime engines",
                        "--permissions=aiplatform.reasoningEngines.get,aiplatform.reasoningEngines.query",
                        "--stage=GA", "--quiet"], check=True, capture_output=True)
        print("✅ Rolle angelegt:", A2A_ROLE)
    subprocess.run(["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID, f"--member={RESEARCHER_PRINCIPAL}",
                    f"--role={A2A_ROLE}", "--condition=None", "--quiet", "--format=none"], check=True)
    print("✅ a2aCaller für den Rechercheur (gilt für alle Engines im Projekt)")
    print("IAM-Propagation dauert ein bis zwei Minuten.")

In [ ]:
# @title B6. Test mit Recht: du → Rechercheur (deployed) → A2A → Analyst (deployed)
############################################
# Erst nach B5 mit A2A_GRANT = True; sonst 403. Erwartet dann echte Zahlen vom Analysten.
# Noch 403? Ein bis zwei Minuten warten und erneut ausführen.

await ask_with_tools(client.agent_engines.get(name=RESEARCHER_ENGINE_NAME), RESEARCH_QUESTION)

In [ ]:
# @title B7. Audit: Wer hat BigQuery abgefragt?
############################################
# Erwartet: nur die Identität des Analysten (und du aus dem lokalen Test A3), nie der Rechercheur.
# Der Rechercheur hat kein BigQuery-Recht; er erreicht die Daten nur über den Analysten.

AUDIT_SQL = """
SELECT creation_time, principal_subject, state
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
  AND statement_type = 'SELECT'
ORDER BY creation_time DESC
LIMIT 10
"""

analyst_id = ANALYST_ENGINE_NAME.split("/")[-1]
for job in bigquery.Client(project=PROJECT_ID).query(AUDIT_SQL).result():
    who = "Analyst (Agent Identity)" if analyst_id in job.principal_subject else job.principal_subject
    print(f"{job.creation_time:%Y-%m-%d %H:%M}  {job.state:5}  {who}")

##### <font color="blue">*Ergebnis (2026-10-26, von Claude durchgespielt)*

| Schritt | Ergebnis |
|---|---|
| A3: Analyst lokal als A2A-Server | Task `TASK_STATE_COMPLETED`, Tool-Aufruf in der History, echte Zahlen (Jennifer 58,385 · Amanda 35,818 · Jessica 33,920) |
| A6: du → deployter Analyst | Agent Card gelesen, Task `COMPLETED`, dieselben Zahlen; BigQuery lief als Identität des Analysten |
| B2: Rechercheur lokal → deployter Analyst | funktioniert; der Analyst hat von sich aus die Spalte „alle Geschlechter“ ergänzt (ein Kollege, kein Werkzeug) |
| B4: deployter Rechercheur, ohne Recht | zuerst **401** (eigener httpx-Client ohne Zertifikat der Runtime), nach dem mTLS-Umbau in B1 **403** `aiplatform.reasoningEngines.query`: angemeldet, aber nicht berechtigt |
| B5: Recht nur auf den Analysten | geht nicht: keine IAM-Policy pro Engine, Bedingungen auf `resource.name` wirkungslos (siehe B5) |
| B5 ohne Bedingung | auf Entscheidung des Nutzers ausgeführt: eigene Rolle `a2aCaller` (nur `reasoningEngines.get` und `.query`) auf Projektebene; gilt damit für alle Engines im Projekt |
| B6: deployter Rechercheur → A2A → deployter Analyst | funktioniert, zwei Minuten nach B5. Die Frage nannte kein Geschlecht, also hat der Analyst beide summiert: Jennifer 58,533 · Amanda 35,843 · Jessica 33,988 (echte Werte; nur Mädchen: 58,385 · 35,818 · 33,920) |
| B7: Audit | alle BigQuery-Jobs liefen als Analyst oder als du; der Rechercheur taucht nie auf |

**Was A2A gegenüber UC4 zeigt:** Die Agent Identity kann andere Agents auf Agent Runtime aufrufen, weil deren Endpunkt eine Google-API ist. Zwei Dinge muss man dafür selbst machen: den eigenen HTTP-Client mit dem Zertifikat der Runtime an den mTLS-Endpunkt schicken (B1) und das Aufrufrecht vergeben (B5), das sich derzeit nicht auf eine einzelne Engine beschränken lässt. Wer einzelne Agents gegeneinander abschotten will, braucht dafür getrennte Projekte oder ein Agent Gateway am Eingang (`gateway.ipynb`).

In [ ]:
# @title (Optional) Aufräumen
############################################
# Löscht nur die Ressourcen dieses Notebooks; die Engines der anderen Notebooks bleiben unberührt.
# Reihenfolge: erst die Engines, dann die Rechte ihrer (dann gelöschten) Identitäten, dann Rolle und Staging-Ordner.

# client.agent_engines.delete(name=RESEARCHER_ENGINE_NAME, force=True)
# client.agent_engines.delete(name=ANALYST_ENGINE_NAME, force=True)
# !gcloud projects remove-iam-policy-binding {PROJECT_ID} --member="{ANALYST_PRINCIPAL}" --role="roles/bigquery.jobUser" --condition=None --quiet > /dev/null
# !gcloud projects remove-iam-policy-binding {PROJECT_ID} --member="{RESEARCHER_PRINCIPAL}" --role="projects/{PROJECT_ID}/roles/a2aCaller" --all --quiet > /dev/null
# !gcloud iam roles delete a2aCaller --project={PROJECT_ID} --quiet
# !gcloud storage rm -r {STAGING_BUCKET}/{GCS_DIR}

##### <font color="blue">*Troubleshooting*

| Symptom | Ursache / Fix |
|---|---|
| 404 „Publisher model `…/locations/us-central1/…/gemini-3.8-flash` was not found“ im Analysten | `A2aAgent.set_up()` setzt `GOOGLE_CLOUD_LOCATION` auf die Region der Engine; Gemini 3.8 Flash gibt es nur auf `global`. Fix im Executor-Builder (A2) |
| Task mit Status `TASK_STATE_FAILED`, im Text 429 `RESOURCE_EXHAUSTED` | Gemini-Kontingent kurz erschöpft; später erneut ausführen. Der Task meldet den Fehler, statt eine Antwort zu erfinden |
| 401 Unauthorized, wenn der deployte Rechercheur den Analysten ruft | Das Token der Agent Identity gilt nur mit dem Zertifikat ihrer Runtime (Context-Aware Access). Eigene HTTP-Clients brauchen `mtls.get_default_ssl_context()` und den Endpunkt `…-aiplatform.mtls.googleapis.com` (B1); Google-Bibliotheken wie BigQuery machen das selbst |
| 403 „Permission 'aiplatform.reasoningEngines.query' denied“ | Dem Rechercheur fehlt das Aufrufrecht (B5), oder es wirkt noch nicht (ein bis zwei Minuten warten). `agentDefaultAccess` enthält es nicht; schon das Lesen der Agent Card braucht `query` |
| `UnknownApiResponseError: Failed to parse response as JSON … Extra data` beim Test des deployten Agents | Ein Tool-Ergebnis mit geschweiften Klammern (z. B. ein JSON-Fehlertext) bringt den Stream-Parser des SDK durcheinander; nur den Text der Meldung zurückgeben (B1) |
| `getIamPolicy` auf die Engine liefert 404 | Engines haben keine eigene IAM-Policy; das Recht auf eine einzelne Engine geht nur über eine IAM-Bedingung auf `resource.name` (B5) |